# Extraction du code NAF #

In [18]:
import pandas as pd
import numpy as np
df = pd.read_csv("extraction/output_icpe_clean.csv", dtype={'entete_siret': str})
# Ajustement des types de données pour afficher le numéro correctement
df = df.dropna(subset=['situation_administrative_code_rubrique'])
num_etab = df["num_etablissement"].astype(str)
df['situation_administrative_code_rubrique'] = df['situation_administrative_code_rubrique'].astype(int)
df['num_etablissement'] = num_etab.str.rjust(10, '0')
df.to_csv("output_icpe_clean.csv", index=False)
df.head()

,num_etablissement,entete_nom,entete_siret,entete_etat_d_activite,entete_regime_en_vigueur_de_l_etablissement_2,entete_statut_seveso,entete_ied_-_mtd,derniere_date_inspection,situation_administrative_code_rubrique,situation_administrative_alinea,situation_administrative_libelle_rubrique,situation_administrative_regime_autorise,situation_administrative_volume
0,0000000039,COLLECTES VALORISATION ENERGIE DECHETS - COVED,34340353103542,En exploitation avec titre,Autorisation,Non Seveso,Oui,2026-01-26,2782,NaN,Autres traitements biologiques de déchets non ...,Autorisation,NaN
1,0000000039,COLLECTES VALORISATION ENERGIE DECHETS - COVED,34340353103542,En exploitation avec titre,Autorisation,Non Seveso,Oui,2026-01-26,3532,NaN,Valorisation de déchets non dangereux,Autorisation,518.000 t/j
2,0000000002,PITCH IMMO SNC : HEXAHUB : hangars logistiques...,42298971500186,En exploitation avec titre,Autorisation,Non Seveso,Non,NaN,1436,2,Liquides combustibles de point éclair compris ...,Déclaration avec contrôle,900.000 t
3,0000000002,PITCH IMMO SNC : HEXAHUB : hangars logistiques...,42298971500186,En exploitation avec titre,Autorisation,Non Seveso,Non,NaN,1450,1,Solides inflammables,Autorisation,150.000 t
4,0000000002,PITCH IMMO SNC : HEXAHUB : hangars logistiques...,42298971500186,En exploitation avec titre,Autorisation,Non Seveso,Non,NaN,1510,1,Entrepôts couverts soumis à EE systématique,Autorisation,NaN


In [19]:
# Merge sur le siret pour récupérer le code NAF
# ATTENTION : tourne en 1m30 environ
with open('extraction/StockEtablissement_utf8.csv', 'r', encoding='utf-8') as f:
    print(f.readline())
columns_to_use = ['siret', 'activitePrincipaleEtablissement', 'nomenclatureActivitePrincipaleEtablissement']
df_NAF = pd.read_csv('extraction/StockEtablissement_utf8.csv', usecols=columns_to_use)
df_NAF.rename(columns={'siret': 'entete_siret'}, inplace=True)
df_NAF['entete_siret'] = df_NAF['entete_siret'].astype(str)
df['entete_siret'] = df['entete_siret'].astype(str)

#on ouvre la base siren et on extrait les colonnes intéressantes
columns_to_use = ['codePostalEtablissement', 'coordonneeLambertAbscisseEtablissement','coordonneeLambertOrdonneeEtablissement', 'siret']
#siren = pd.read_csv('StockEtablissement_utf8.csv', usecols=columns_to_use, sep=',')

dtype = {
    'codePostalEtablissement': 'str',  # ou 'str' si format texte (ex: "75001")
    'coordonneeLambertAbscisseEtablissement': 'Float64',
    'coordonneeLambertOrdonneeEtablissement': 'Float64'
}

siren = pd.read_csv(
    'extraction/StockEtablissement_utf8.csv',
    usecols=columns_to_use,
    dtype=dtype,
    sep=',',
    low_memory=True,  # évite l'inférence de type par blocs
    na_values=['[ND]']  # Remplace "[ND]" par NaN
)
siren.head() 

siren,nic,siret,statutDiffusionEtablissement,dateCreationEtablissement,trancheEffectifsEtablissement,anneeEffectifsEtablissement,activitePrincipaleRegistreMetiersEtablissement,dateDernierTraitementEtablissement,etablissementSiege,nombrePeriodesEtablissement,complementAdresseEtablissement,numeroVoieEtablissement,indiceRepetitionEtablissement,dernierNumeroVoieEtablissement,indiceRepetitionDernierNumeroVoieEtablissement,typeVoieEtablissement,libelleVoieEtablissement,codePostalEtablissement,libelleCommuneEtablissement,libelleCommuneEtrangerEtablissement,distributionSpecialeEtablissement,codeCommuneEtablissement,codeCedexEtablissement,libelleCedexEtablissement,codePaysEtrangerEtablissement,libellePaysEtrangerEtablissement,identifiantAdresseEtablissement,coordonneeLambertAbscisseEtablissement,coordonneeLambertOrdonneeEtablissement,complementAdresse2Etablissement,numeroVoie2Etablissement,indiceRepetition2Etablissement,typeVoie2Etablissement,libelleVoie2Etablissement,codePostal2Etablissement,l

,siret,codePostalEtablissement,coordonneeLambertAbscisseEtablissement,coordonneeLambertOrdonneeEtablissement
0,32517500016,98770,<NA>,<NA>
1,32517500024,84000,851150.098259,6317267.146095
2,32517500032,84000,848084.236681,6316548.347115
3,32517500040,13420,913005.278239,6245869.653506
4,32517500057,13004,894589.970053,6247476.026088


In [20]:
# ATTENTION : tourne en 13m environ
df = df.merge(df_NAF, on='entete_siret', how='left')
df.rename(columns={'activitePrincipaleEtablissement': 'code_naf'}, inplace=True)
siren.drop(columns=['codePostalEtablissement'], inplace=True)
siren.rename(columns={'siret': 'entete_siret'}, inplace=True)
siren['entete_siret'] = siren['entete_siret'].astype(str)
df = df.merge(siren, on='entete_siret', how='left')

In [21]:
df.to_csv("output_icpe_with_NAF.csv", index=False)
print("Le CSV a été nettoyé et enregisté avec succès ! ;)")

Le CSV a été nettoyé et enregisté avec succès ! ;)


# Organisation de la base de données #

In [22]:
df.rename(columns={'entete_nom': 'Nom',
                   'entete_siret': 'Siret',
                   'situation_administrative_code_rubrique': 'Code_Rubrique',
                   'entete_statut_seveso': 'Statut_Seveso',
                   'situation_administrative_libelle_rubrique': 'Libelle_rubrique',
                   'situation_administrative_volume' : 'Volume'}, inplace=True)
df = df.drop(columns=['entete_etat_d_activite', 'entete_regime_en_vigueur_de_l_etablissement_2', 'entete_ied_-_mtd', 'situation_administrative_alinea'])
df.head()

,num_etablissement,Nom,Siret,Statut_Seveso,derniere_date_inspection,Code_Rubrique,Libelle_rubrique,situation_administrative_regime_autorise,Volume,code_naf,nomenclatureActivitePrincipaleEtablissement,coordonneeLambertAbscisseEtablissement,coordonneeLambertOrdonneeEtablissement
0,0000000039,COLLECTES VALORISATION ENERGIE DECHETS - COVED,34340353103542,Non Seveso,2026-01-26,2782,Autres traitements biologiques de déchets non ...,Autorisation,NaN,38.32Z,NAFRev2,<NA>,<NA>
1,0000000039,COLLECTES VALORISATION ENERGIE DECHETS - COVED,34340353103542,Non Seveso,2026-01-26,3532,Valorisation de déchets non dangereux,Autorisation,518.000 t/j,38.32Z,NAFRev2,<NA>,<NA>
2,0000000002,PITCH IMMO SNC : HEXAHUB : hangars logistiques...,42298971500186,Non Seveso,NaN,1436,Liquides combustibles de point éclair compris ...,Déclaration avec contrôle,900.000 t,41.10A,NAFRev2,651515.228071,6863554.587773
3,0000000002,PITCH IMMO SNC : HEXAHUB : hangars logistiques...,42298971500186,Non Seveso,NaN,1450,Solides inflammables,Autorisation,150.000 t,41.10A,NAFRev2,651515.228071,6863554.587773
4,0000000002,PITCH IMMO SNC : HEXAHUB : hangars logistiques...,42298971500186,Non Seveso,NaN,1510,Entrepôts couverts soumis à EE systématique,Autorisation,NaN,41.10A,NAFRev2,651515.228071,6863554.587773
